# Drag-and-drop FITS processing (no helper service)

This notebook demonstrates processing images that are **drag-and-dropped into
the JupyterLab file browser** — no local helper service required.

How it works: uploads land in the browser's IndexedDB storage, the kernel sees
them immediately via the mounted contents drive, and deleting a file from the
kernel really frees the storage. So a polling loop can process each image as it
finishes uploading and delete it right away — only a file or two ever exists in
browser storage at a time.

**Demo recipe**

1. Run all cells. Along the way, two cells run **direct network tests** of the
   bandaid pipeline's only two outbound calls — the Gaia-DR2-via-VizieR cone
   search and the HuggingFace download of the Ballet CNN weights; both
   services send CORS headers, so no proxy is needed for either — and another
   cell installs the folder-only drop guard. The last cell starts `watch()`,
   which polls the `incoming/` folder once a second.
2. In the file browser, open `incoming/`, then drag a **folder** of FITS files
   into it from your file manager. (A folder is required, not just better:
   files inside a dropped folder upload one at a time, so storage stays
   bounded, while loose files dropped together all upload in parallel. The
   drop-guard cell installs a page-level guard that rejects loose-file drops
   on the file browser with a warning; it is active once that cell has run.)
3. Watch one line print per image — filename plus a few header cards, with
   ~1 s of simulated photometry — and watch each file disappear from the file
   browser as soon as it is processed.

**Stopping the loop**

- Create a file named `STOP` inside `incoming/` (file browser: right-click →
  New File, rename to `STOP`), or
- wait for the idle timeout (default 120 s with no new files), or
- the notebook stop button, if kernel interrupt works in your deployment.


In [ ]:
# Drive-semantics smoke test: the kernel's filesystem is the JupyterLite
# contents drive, so listings/reads/deletes go through the live contents
# manager (IndexedDB), not a private kernel filesystem.
import os
import sys
import time

print("platform:", sys.platform)
print("cwd:     ", os.getcwd())
print("contents:", sorted(os.listdir(".")))

t0 = time.monotonic()
time.sleep(2)
print(f"time.sleep(2) took {time.monotonic() - t0:.2f}s")

# Manual checks (do these once per deployment, in the file browser UI):
# 1. Create a new text file at the drive root -> re-run this cell; it must
#    appear in the listing WITHOUT a kernel restart.
# 2. Run: open("kernel_test.txt", "w").write("x")  -> it must appear in the
#    file browser. Then os.remove("kernel_test.txt") -> it must disappear,
#    including from devtools > Application > IndexedDB > JupyterLite Storage.
# 3. Run time.sleep(30) and press the stop button to learn whether kernel
#    interrupt works here; the watch loop below does not depend on it.


In [ ]:
# Direct astroquery test - no proxy, no helper service. The only astroquery
# call the bandaid pipeline makes is a Gaia DR2 cone search *via VizieR*
# (Vizier.query_region on catalog I/345/gaia2, brightest-first, row-limited;
# see bandaid's catalog.cached_gaia_radecs). VizieR sends
# Access-Control-Allow-Origin: *, so the query should work straight from the
# browser with only the pyodide-http patch - no CORS proxy needed.
import os
os.environ.setdefault("PYTHON_KEYRING_BACKEND", "keyring.backends.null.Keyring")

import pyodide_http
pyodide_http.patch_all()

import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.vizier import Vizier

# Field center of the Qatar-8 test frames (RA/DEC from their headers);
# radius ~ half the Seestar S50 field of view, as bandaid computes it.
center = SkyCoord(ra=157.925 * u.deg, dec=70.413 * u.deg)
vizier = Vizier(
    columns=["+Gmag", "RA_ICRS", "DE_ICRS", "pmRA", "pmDE"],  # "+Gmag" = brightest first
    row_limit=1000,
)
result = vizier.query_region(center, radius=0.36 * u.deg, catalog="I/345/gaia2")
table = result[0]
print(f"Gaia DR2 via VizieR, no proxy: {len(table)} sources, "
      f"brightest Gmag = {table['Gmag'].min():.2f}")
table[:5]


In [ ]:
# Direct HuggingFace download test - the only other network call the bandaid
# pipeline makes: eloy's Ballet centroider fetches its CNN weights on first
# use via hf_hub_download(repo_id="lgrcia/ballet",
# filename="centroid_15x15.npz") (eloy/ballet/model.py, download_weights).
# Probed with curl + an Origin: header (2026-07-28): both hops are CORS-clean.
#   huggingface.co/.../resolve/main/... -> 302 with
#     Access-Control-Allow-Origin echoing the Origin, and
#     Access-Control-Expose-Headers listing ETag, X-Linked-ETag and
#     X-Repo-Commit (the headers hf_hub_download reads);
#   the CDN it redirects to (us.aws.cdn.hf.co) -> 200 with
#     Access-Control-Allow-Origin: *.
# Browsers demand CORS approval on every redirect hop (the AAVSO lesson), and
# here every hop passes - so the ~39 MB weights file should download straight
# from the browser with only the pyodide-http patch.
#
# huggingface_hub itself is not in this environment, so the network path is
# tested with plain requests; if huggingface_hub is importable the real
# hf_hub_download is tried too. Note for bundling the library later:
# huggingface_hub >= 1.0 switched from requests to httpx, which pyodide-http
# does NOT patch - an in-browser hf_hub_download needs the 0.x series.
import io
import os

import numpy as np
import pyodide_http
import requests

pyodide_http.patch_all()  # no-op if the astroquery cell already ran

# Disabling the native xet accelerator is what bandaid does for cosmetic
# reasons (scripts._quiet_hf_xet); in wasm it is a hard requirement, since
# the hf_xet extension cannot exist here.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

URL = "https://huggingface.co/lgrcia/ballet/resolve/main/centroid_15x15.npz"

resp = requests.get(URL, timeout=120)
resp.raise_for_status()
weights = np.load(io.BytesIO(resp.content))
print(f"plain requests: {len(resp.content) / 1e6:.1f} MB from {resp.url.split('/')[2]}")
print(f"npz contains {len(weights.files)} arrays, e.g. "
      + ", ".join(f"{k}{weights[k].shape}" for k in weights.files[:3]))

try:
    from huggingface_hub import hf_hub_download
except ImportError:
    print("huggingface_hub not installed - library-level test skipped "
          "(the raw download above is the CORS answer).")
else:
    path = hf_hub_download(repo_id="lgrcia/ballet", filename="centroid_15x15.npz")
    print(f"hf_hub_download cached to: {path}")


In [ ]:
# Folder-only drop guard. The kernel runs in a web worker with no DOM access,
# but JupyterLab executes application/javascript outputs on the main thread,
# so this cell can install a page-level drop filter: a capture-phase listener
# that runs before JupyterLab's own drop handler and rejects any drop on the
# file browser that isn't purely folders (loose files upload in parallel and
# defeat the bounded-storage behavior). Entries are only inspectable during
# 'drop', not 'dragover', so the drag cursor still shows "copy" - the drop is
# refused on release, with a toast explaining why. Active only after this
# cell has run; harmless to re-run (guarded by a window flag).
from IPython.display import Javascript, display

FOLDER_ONLY_JS = """
(() => {
  if (window._folderOnlyDropGuard) { return; }
  window._folderOnlyDropGuard = true;
  const toast = (msg) => {
    const div = document.createElement('div');
    div.textContent = msg;
    Object.assign(div.style, {
      position: 'fixed', top: '12px', left: '50%',
      transform: 'translateX(-50%)', zIndex: 10000,
      background: 'var(--jp-warn-color1, #f57c00)', color: 'white',
      padding: '8px 16px', borderRadius: '4px',
      font: '13px var(--jp-ui-font-family, sans-serif)',
      boxShadow: '0 2px 8px rgba(0,0,0,0.3)',
    });
    document.body.appendChild(div);
    setTimeout(() => div.remove(), 5000);
  };
  document.addEventListener('drop', (ev) => {
    const onListing = ev.target instanceof Element && ev.target.closest('.jp-DirListing');
    if (!onListing) { return; }
    const items = Array.from(ev.dataTransfer?.items ?? []).filter((i) => i.kind === 'file');
    if (items.length === 0) { return; }  // not a native file drag
    const entries = items.map((i) => i.webkitGetAsEntry && i.webkitGetAsEntry());
    if (entries.every((e) => e && e.isDirectory)) { return; }  // all folders: allow
    ev.preventDefault();
    ev.stopImmediatePropagation();  // JupyterLab's upload handler never runs
    toast('Loose files rejected - drop a single folder of images instead.');
  }, true);  // capture phase: fires before the DirListing drop handler
})();
"""

display(Javascript(FOLDER_ONLY_JS))
print("Folder-only drop guard installed: dropping loose files on the file "
      "browser is now rejected with a warning; folders are accepted.")


In [ ]:
import os
import time
from astropy.io import fits

WATCH_DIR = "incoming"
FITS_EXTS = (".fit", ".fits", ".fts")
CARDS = ("DATE-OBS", "EXPTIME", "FILTER", "OBJECT", "IMAGETYP")  # print those present
MAX_RETRIES = 5  # consecutive fits.open failures before giving up on a file


def fits_candidates(root):
    """All FITS files under root, any depth (folder drops create subdirs)."""
    for dirpath, _dirnames, filenames in os.walk(root):
        for name in sorted(filenames):
            if name.lower().endswith(FITS_EXTS):
                yield os.path.join(dirpath, name)


def process_one(path):
    """PoC 'photometry': print filename + selected header cards, ~1 s of work."""
    t0 = time.monotonic()
    # memmap=False: the drive mount buffers whole files, no mmap support.
    # mode='readonly': never write the file back through the drive on close.
    with fits.open(path, mode="readonly", memmap=False) as hdul:
        hdr = hdul[0].header
        cards = "  ".join(f"{c}={hdr[c]!r}" for c in CARDS if c in hdr)
    time.sleep(1.0)  # simulate per-image photometry cost
    print(f"[done {time.monotonic() - t0:4.1f}s] {path}  {cards}")


def prune_empty_dirs(root):
    """Remove emptied dropped-folder directories (bottom-up), never root itself."""
    for dirpath, dirnames, filenames in os.walk(root, topdown=False):
        if dirpath != root and not dirnames and not filenames:
            try:
                os.rmdir(dirpath)
            except OSError:
                pass


def watch(watch_dir=WATCH_DIR, poll=1.0, idle_timeout=120):
    os.makedirs(watch_dir, exist_ok=True)
    stop_file = os.path.join(watch_dir, "STOP")
    done, failures, last_size = set(), {}, {}
    n_ok = 0
    last_activity = time.monotonic()
    print(f"Watching {watch_dir!r}. Drop a folder of FITS files into it in the "
          f"file browser. To stop: create a file named STOP in {watch_dir!r} "
          f"(or wait {idle_timeout}s idle).")
    try:
        while True:
            if os.path.exists(stop_file):
                os.remove(stop_file)
                print("STOP file seen - exiting.")
                break
            for path in list(fits_candidates(watch_dir)):
                if path in done:
                    continue
                try:
                    size = os.path.getsize(path)
                except OSError:
                    continue  # vanished between walk and stat
                # Partial-upload guard: complete only when nonzero, unchanged
                # since last poll, and a whole number of 2880-byte FITS blocks
                # (mid-upload sizes are 1 MiB multiples, which fail this).
                complete = (size > 0
                            and last_size.get(path) == size
                            and size % 2880 == 0)
                last_size[path] = size
                last_activity = time.monotonic()  # new or growing file = activity
                if not complete:
                    continue
                try:
                    process_one(path)
                except Exception as exc:
                    failures[path] = failures.get(path, 0) + 1
                    if failures[path] < MAX_RETRIES:
                        continue  # maybe a stalled upload resumed; retry next poll
                    print(f"[skip] {path}: unreadable after "
                          f"{MAX_RETRIES} tries ({exc}); left in place")
                    done.add(path)
                    continue
                done.add(path)
                n_ok += 1
                os.remove(path)          # free IndexedDB immediately
                last_size.pop(path, None)
                last_activity = time.monotonic()
            prune_empty_dirs(watch_dir)
            if idle_timeout and time.monotonic() - last_activity > idle_timeout:
                print(f"No activity for {idle_timeout}s - exiting.")
                break
            time.sleep(poll)
    except KeyboardInterrupt:
        print("Interrupted - exiting.")
    print(f"Processed {n_ok} file(s); {len(done) - n_ok} skipped.")


watch()
